# 多目标建模
- 基础结构演进
    - Shared-Bottom 
    - MMOE
    - PLE
- 任务依赖建模
    - ESMM
    - ESM2
- 多目标损失融合
    - Uncertainty Weight：基于不确定性的自适应加权
    - GradNorm：梯度标准化方法
    - Pareto Optimization：帕累托优化框架
- 总结

## MMOE（Multi-gate Mixture-of-Experts）
![](resource/MMOE.png)

针对 Shared-Bottom 对相关性低的多个任务出现负迁移的现象，OMoE（OneGate Mixture-of-Experts）将底层共享的一个 Shared-Bottom 模块拆分成了多个 Expert，最终 OMoE 的输出为多个 Expert 的加权和，本质可以看成是专家网络和全局门控的双层结构。

虽然 OMoE 通过底层多专家融合的方式提升了特征表征的多样性，从最终的实验结果看，确实可以一定程度上缓解低相关性任务的负迁移问题，但没有彻底解决多任务冲突的问题。因为不同任务反向传播的梯度还是会直接影响底层专家网络的学习。

为了进一步缓解多任务冲突，MMoE（Multi-gate Mixture-of-Experts）:cite:`ma2018modeling` 为每个任务配备专属门控网络，实现了门控从“全局共享”升级为“任务自适应”的方式。MMoE 的数学表达式可以表示为：

$$
\begin{aligned}
\mathbf{e}_k &= f_k(\mathbf{x}) \\
g_t(\mathbf{x}) &= \text{softmax}(\mathbf{W}_t \mathbf{x}) \\
\mathbf{h}_t &= \sum_{k=1}^{K} g_{t,k} \cdot \mathbf{e}_k \\
\hat{y}_t &= f_t(\mathbf{h}_t)
\end{aligned}
\tag{3.4.2}
$$

其中 $\mathbf{x}$ 表示底层的特征输入，$\mathbf{e}_k$ 表示第 $k$ 个专家网络的输出，$g_t(\mathbf{x})$ 表示第 $t$ 个任务融合专家网络的门控向量，$\mathbf{h}_t$ 表示第 $t$ 个任务融合专家网络的输出，$\hat{y}_t$ 表示第 $t$ 个任务的预测结果。

差异化特征融合门控网络 $g_t$ 根据任务特性选择专家组合，例如在电商场景，CTR 任务门控加权“即时兴趣”“价格敏感”专家，CVR 任务门控：侧重“消费能力”“品牌忠诚”专家。

当任务 $i$ 与 $j$ 冲突时，MMoE 的门控机制会让两个任务学习到不同专家的权重分布。例如，某个专家 $e_m$ 可能在任务 $i$ 的门控网络中获得很高的权重 $g_{i,m}$，而在任务 $j$ 的门控网络中获得很低的权重 $g_{j,m}$。这样一来，专家 $e_m$ 的参数更新主要由任务 $i$ 的梯度决定，而任务 $j$ 的梯度影响很小，从而实现了梯度隔离。不同任务通过选择不同的专家组合，可以各自学习到适合自己的特征表示，缓解任务冲突。

### MMoE 代码实践

MMoE 模型构建代码如下，先组装输入到 MoE 网络中的特征 `dnn_inputs`，然后为每个任务创建一个门控网络输出最终融合专家网络的门控向量。最后为每个任务都创建一个任务塔，并且不同任务塔的输入都是对应任务的门控向量和多个专家网络融合后的向量。

In [ ]:
## 错的，大体结构可以看，但后续需要修改
import torch
import torch.nn as nn
import torch.nn.functional as F

class Expert(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_rate):
        super(Expert,self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Dropout(dropout_rate) # 在专家内部加dropout
        )
    def forward(self, X):
        return self.network(X)

class Gate(nn.Module):
    def __init__(self, input_size, hidden_size, expert_num):
        super(Gate, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, expert_num)
        )
    def forward(self,X):
        return self.network(X)

class MMOE(nn.Module):
    def __init__(self, input_size, expert_num, task_num, hidden_size, temperature):
        super(MMOE,self).__init__()
        self.input_size = input_size
        self.expert_num = expert_num
        self.task_num = task_num
        self.hidden_size = hidden_size
        self.temperature = temperature
        
        self.experts = nn.ModuleList([nn.Linear(input_size, hidden_size) 
                                      for _ in range(expert_num)])
        self.gates = nn.ModuleList([nn.Linear(input_size, expert_num) 
                                    for _ in range(task_num)])
        self.fcs = nn.ModuleList([nn.Linear(hidden_size,1) 
                                  for _ in range(task_num)])
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax()
    
    def forward(self, X):
        expert_output = []
        gate_outputs = []
        for i in range(self.expert_num):
            expert_output.append(self.relu(self.experts[i](X)))
            
        for i in range(self.task_num):
            gate_outputs.append(self.softmax(self.gates[i](X) / self.temperature, dim=1))
            
        expert_output = torch.stack(expert_output)
        gate_outputs = torch.stack(gate_outputs)
        res = []
        for i in range(self.task_num):
            tmp = torch.zeros(self.expert_num)
            for j in range(self.expert_num):
                tmp += gate_outputs[i][j] * expert_output[j]
            res.append(tmp)
        
        res = torch.stack(res)
        out = []
        for i in range(self.task_num):
            out.append(self.fcs[i](res[i]))
        
        out = torch.stack(out)
        return out

### 缓解 MMoE 专家极化现象的三大核心策略

在多任务学习中，**MMoE（Multi-gate Mixture-of-Experts）** 虽然通过任务专属门控缓解了任务冲突，但仍面临 **专家极化（Expert Polarization）** 问题：少数专家被高频使用，其余专家几乎“闲置”。为提升专家多样性与模型鲁棒性，以下三种策略被广泛采用并验证有效。

---

#### 1. Dropout 插入策略：增强专家泛化能力

##### 🎯 目标  
防止专家网络对特定任务过拟合，鼓励其学习更通用的特征表示。

##### ✅ 推荐位置  
**在每个 Expert 网络内部的 MLP 层之间插入 Dropout**，而非在门控输出或融合后向量上。

##### 💡 原理  
- Dropout 随机屏蔽神经元，迫使网络不依赖单一路径；
- 在 Expert 内部使用，可抑制“强势专家”对特定模式的强记忆；
- 提升多个专家之间的功能互补性。

##### 💻 示例代码（PyTorch）
```python
class Expert(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout_rate=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),  # ← 关键：Expert 内部加 Dropout
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )

    def forward(self, x):
        return self.net(x)

### 2. 负载均衡损失（Load Balancing Loss）

#### 🎯 目标  
鼓励每个任务的门控网络（gate）**均匀地使用所有专家**，避免某些专家被频繁调用而其他专家“闲置”，从而缓解专家极化。

#### 🔍 原理  
在训练过程中，除了主任务损失外，额外引入一个辅助损失（auxiliary loss），该损失衡量：
- 各专家被使用的频率是否均衡；
- 不同任务对专家的依赖是否过于集中。

通过最小化该损失，模型会主动调整门控权重，使专家资源得到更公平的分配。

#### 📐 数学表达  
设共有 $K$ 个专家、$T$ 个任务，$f_{t,k}$ 表示任务 $t$ 对专家 $k$ 的平均门控权重（在 batch 或 epoch 上统计），则负载均衡损失可定义为：

$$
\mathcal{L}_{\text{aux}} = \left( \frac{1}{T} \sum_{t=1}^{T} \sum_{k=1}^{K} f_{t,k} \right) \cdot \left( \frac{1}{K} \sum_{k=1}^{K} \left( \frac{1}{T} \sum_{t=1}^{T} f_{t,k} \right)^2 \right)
$$

> 💡 实际中常简化为：  
> $$
> \mathcal{L}_{\text{aux}} = \sum_{k=1}^{K} \left( \frac{1}{B} \sum_{i=1}^{B} g_{i,k} \right) \cdot \left( \frac{1}{B} \sum_{i=1}^{B} g_{i,k} \right)
> = \| \text{mean}_B(g_k) \|_2^2
> $$  
> 其中 $g_{i,k}$ 是样本 $i$ 在某任务门控中对专家 $k$ 的权重，$B$ 为 batch size。

#### 💻 PyTorch 实现示例
```python
# 假设 gate_weights: [B, K] 是某个任务的门控输出
expert_importance = gate_weights.mean(dim=0)  # [K]
load_balance_loss = (expert_importance ** 2).sum()  # L2 norm squared

# 总损失 = 主任务损失 + λ * load_balance_loss
total_loss = task_loss + 0.01 * load_balance_loss



### 3. Softmax 温度控制（Temperature Scaling）

#### 🎯 目标  
通过引入温度参数 $T$ 调节 Softmax 输出的“锐度”，使 MMoE 中门控网络（gate）对专家的选择更加平滑，从而缓解专家极化（expert polarization）问题。

---

#### 🔍 原理

标准 Softmax 函数定义为：

$$
\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

引入温度参数 $T > 0$ 后，变为 **带温度的 Softmax**：

$$
\text{softmax}_T(z_i) = \frac{e^{z_i / T}}{\sum_{j=1}^{K} e^{z_j / T}}
$$

##### 温度 $T$ 的影响：
|
 温度值 
|
 输出分布特性 
|
 对 MMoE 的影响 
|
|
--------
|
--------------
|
----------------
|
|
 $T = 1$ 
|
 标准 Softmax 
|
 默认行为 
|
|
 $T > 1$ 
|
 分布更
**
平滑
**
（soft），各专家权重差异减小 
|
 ✅ 缓解极化，鼓励多专家参与 
|
|
 $T < 1$ 
|
 分布更
**
尖锐
**
（sharp），集中在 top-1 或少数专家 
|
 ❌ 加剧极化 
|

> 💡 在 MMoE 中，**适当提高温度（如 $T = 1.5 \sim 3.0$）可有效防止门控过度偏向某些专家**。

---

#### 💻 PyTorch 实现示例

```python
import torch
import torch.nn.functional as F

# 假设 gate_logits 形状为 [batch_size, num_experts]
gate_logits = torch.randn(32, 8)  # 示例数据

temperature = 2.0  # >1.0 用于平滑分布

# 应用带温度的 Softmax
gate_weights = F.softmax(gate_logits / temperature, dim=-1)

print(gate_weights.sum(dim=-1))  # 仍为 1.0（概率分布）

## CGC/PLE（Customized Gate Control）/ (Progressive Layered Extraction) 
![](resource/CGC.png)

## 任务依赖建模ESMM/ESM2

[ESMM/ESM2](https://datawhalechina.github.io/fun-rec/chapter_2_ranking/4.multi_objective/2.dependency_modeling.html)

[阿里CVR预估模型之ESMM](https://zhuanlan.zhihu.com/p/57481330)